# **Exercise 1: Implementing the LoRALayer**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- LoRALayer ----------
class LoRALayer(nn.Module):
    """
    Low-rank adapter: produces a residual x @ A @ B * (alpha/rank)
    Shapes:
      x: [batch, in_dim]
      A: [in_dim, rank]
      B: [rank, out_dim]
      out: [batch, out_dim]
    """
    def __init__(self, in_dim, out_dim, rank=4, alpha=8):
        super().__init__()
        assert rank > 0, "rank must be > 0"
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Init A ~ N(0, 1/sqrt(rank)), B = 0  → LoRA starts as "do nothing"
        std_dev = 1.0 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))

    def forward(self, x):
        # x @ A: compress features to rank dims
        # (x @ A) @ B: expand back to out_dim
        # scaling keeps training stable
        return (x @ self.A @ self.B) * self.scaling

# **Exercise 2: Implementing the LinearWithLoRA Layer**

In [16]:
# ---------- LinearWithLoRA ----------
class LinearWithLoRA(nn.Module):
    def __init__(self, linear: nn.Linear, rank=4, alpha=8):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )
        # Ensure LoRA parameters are on the same device as the original linear layer
        self.lora.to(self.linear.weight.device)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [3]:
# Hyperparameters for the tiny test
random_seed = 123
torch.manual_seed(random_seed)

# original linear
base_linear = nn.Linear(5, 3)  # in=5, out=3
x = torch.randn(2, 5)

print("Input x:\n", x)
print("\nBase Linear:\n", base_linear)
print("\nOriginal output:\n", base_linear(x))

# Apply LoRA on top (initially should match base since LoRA starts at zero)
layer_lora_1 = LinearWithLoRA(base_linear, rank=2, alpha=4)
print("\nLinearWithLoRA output (initial):\n", layer_lora_1(x))

Input x:
 tensor([[ 1.7603,  0.6547,  0.5490,  0.3671,  0.1219],
        [ 0.6466, -1.4168,  0.8429, -0.6307,  1.2340]])

Base Linear:
 Linear(in_features=5, out_features=3, bias=True)

Original output:
 tensor([[-0.5057, -0.2512, -0.2165],
        [-0.9906,  0.4598, -0.4026]], grad_fn=<AddmmBackward0>)

LinearWithLoRA output (initial):
 tensor([[-0.5057, -0.2512, -0.2165],
        [-0.9906,  0.4598, -0.4026]], grad_fn=<AddBackward0>)


# **Exercise 3: Creating a Small Neural Network and Applying LoRA**

In [4]:
# single-layer net, then swap to LoRA-wrapped version
single = nn.Linear(5, 3)
x_test = torch.randn(4, 5)
y0 = single(x_test)

single_lora = LinearWithLoRA(single, rank=2, alpha=4)
y1 = single_lora(x_test)

print("\n[Exercise 3] Outputs equal initially? ",
      torch.allclose(y0, y1, atol=1e-6))


[Exercise 3] Outputs equal initially?  True


# **Exercise 4: Merging LoRA Matrices and Testing Equivalence**

In [5]:
# ---------- LinearWithLoRAMerged ----------
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear: nn.Linear, rank=4, alpha=8):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # A@B is [in_dim, out_dim]; F.linear expects [out_dim, in_dim]
        lora_update = (self.lora.A @ self.lora.B).T * (self.lora.alpha / self.lora.rank)
        combined_weight = self.linear.weight + lora_update
        return F.linear(x, combined_weight, self.linear.bias)

# Equivalence test vs LinearWithLoRA
torch.manual_seed(123)
lin = nn.Linear(5, 3)
lw_lora = LinearWithLoRA(lin, rank=2, alpha=4)
lw_merged = LinearWithLoRAMerged(lin, rank=2, alpha=4)

x_eq = torch.randn(3, 5)
y_a = lw_lora(x_eq)
y_b = lw_merged(x_eq)
print("\n[Exercise 4] Merged equals unmerged? ",
      torch.allclose(y_a, y_b, atol=1e-6))


[Exercise 4] Merged equals unmerged?  True


# **Exercise 5: Implementing a Multilayer Perceptron (MLP) and Replacing Layers with LoRA**

In [6]:
# ---------- 3-layer MLP ----------
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        return self.layers(x)

# Architecture
num_features = 28*28
num_hidden_1 = 256
num_hidden_2 = 128
num_classes  = 10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 1e-3
num_epochs = 2  # keep small for a quick demo

# Base model (no LoRA yet)
model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
).to(DEVICE)

optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print("\nDevice:", DEVICE)
print(model)


Device: cuda
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [7]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64
transform = transforms.ToTensor()  # scales to [0,1]

train_dataset = datasets.MNIST(root='data', train=True,  transform=transform, download=True)
test_dataset  = datasets.MNIST(root='data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('\nImage batch:', images.shape, 'Labels:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 42.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


Image batch: torch.Size([64, 1, 28, 28]) Labels: torch.Size([64])


In [8]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            # flatten 28x28 → 784
            features = features.view(features.size(0), -1).to(device)
            targets  = targets.to(device)
            logits   = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    return 100.0 * correct_pred / num_examples

In [9]:
import time

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(features.size(0), -1).to(device)
            targets  = targets.to(device)

            # forward & loss
            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print(f'Epoch {epoch+1:03d}/{num_epochs:03d} | '
                      f'Batch {batch_idx:03d}/{len(train_loader)} | '
                      f'Loss: {loss.item():.4f}')

        print(f'Epoch {epoch+1:03d}: train acc {compute_accuracy(model, train_loader, device):.2f}%')

    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [10]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy (base): {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch 001/002 | Batch 000/937 | Loss: 2.3097
Epoch 001/002 | Batch 400/937 | Loss: 0.2424
Epoch 001/002 | Batch 800/937 | Loss: 0.1607
Epoch 001: train acc 96.28%
Epoch 002/002 | Batch 000/937 | Loss: 0.2088
Epoch 002/002 | Batch 400/937 | Loss: 0.1190
Epoch 002/002 | Batch 800/937 | Loss: 0.0308
Epoch 002: train acc 97.78%
Total Training Time: 0.49 min
Test accuracy (base): 97.05%


In [11]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy (base): {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch 001/002 | Batch 000/937 | Loss: 0.0502
Epoch 001/002 | Batch 400/937 | Loss: 0.0699
Epoch 001/002 | Batch 800/937 | Loss: 0.0678
Epoch 001: train acc 98.61%
Epoch 002/002 | Batch 000/937 | Loss: 0.0595
Epoch 002/002 | Batch 400/937 | Loss: 0.0160
Epoch 002/002 | Batch 800/937 | Loss: 0.0563
Epoch 002: train acc 98.97%
Total Training Time: 0.47 min
Test accuracy (base): 97.58%


# **Exercise 6: Freezing the Original Linear Layers and Training LoRA**

In [15]:
def replace_linear_with_lora_in_sequential(sequential_module: nn.Sequential, rank=4, alpha=8):
    new_layers = []
    for child in sequential_module.children():
        if isinstance(child, nn.Linear):
            # Create a LinearWithLoRA, passing the original linear layer
            lora_linear_layer = LinearWithLoRA(child, rank=rank, alpha=alpha)
            new_layers.append(lora_linear_layer)
        else:
            new_layers.append(child)
    return nn.Sequential(*new_layers)

# Create a new MLP instance for LoRA
model_lora = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
).to(DEVICE)

# Load the state dict from the already trained base model to initialize model_lora
model_lora.load_state_dict(model.state_dict())

# Replace the nn.Linear layers within model_lora's sequential 'layers' attribute
model_lora.layers = replace_linear_with_lora_in_sequential(model_lora.layers, rank=4, alpha=8)

# Move the entire model_lora to the device again to ensure new LoRA parameters are on the correct device
model_lora.to(DEVICE)

def freeze_linear_layers(module: nn.Module):
    for child in module.children():
        if isinstance(child, nn.Linear):
            for p in child.parameters():
                p.requires_grad = False
        else:
            freeze_linear_layers(child)

# Freeze base Linear weights inside the LoRA model
freeze_linear_layers(model_lora)

# Show which params train
print("\nTrainable flags (True=trainable):")
for name, p in model_lora.named_parameters():
    print(f"{name}: {p.requires_grad}")

# Train only LoRA params
optimizer_lora = torch.optim.Adam(filter(lambda p: p.requires_grad, model_lora.parameters()),
                                  lr=learning_rate)

train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'\nTest accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')
print(f'Test accuracy base model: {compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')


Trainable flags (True=trainable):
layers.0.linear.weight: False
layers.0.linear.bias: False
layers.0.lora.A: True
layers.0.lora.B: True
layers.2.linear.weight: False
layers.2.linear.bias: False
layers.2.lora.A: True
layers.2.lora.B: True
layers.4.linear.weight: False
layers.4.linear.bias: False
layers.4.lora.A: True
layers.4.lora.B: True
Epoch 001/002 | Batch 000/937 | Loss: 0.0060
Epoch 001/002 | Batch 400/937 | Loss: 0.0191
Epoch 001/002 | Batch 800/937 | Loss: 0.0012
Epoch 001: train acc 99.41%
Epoch 002/002 | Batch 000/937 | Loss: 0.0103
Epoch 002/002 | Batch 400/937 | Loss: 0.0147
Epoch 002/002 | Batch 800/937 | Loss: 0.0273
Epoch 002: train acc 99.35%
Total Training Time: 0.50 min

Test accuracy LoRA finetune: 97.92%
Test accuracy base model: 97.58%
Test accuracy LoRA model: 97.92%
